# Este notebook analiza el **dataset estructurado completo generado por el Miner** para TanStack.

El archivo `results.json` integra evidencia de:

- **CodeQL**: hallazgos sobre código fuente.
- **Grype**: vulnerabilidades conocidas en dependencias.
- Metadatos de repositorios y referencias a SBOM generados con Syft.

El objetivo es analizar de manera conjunta y separada las distintas fuentes de evidencia, identificar patrones, concentraciones, severidades y diferencias entre repositorios, y dejar resultados estructurados para el Visualizer.



## 1. Importaciones

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown


## 2. Carga de `results.json`

In [ ]:
import os

_env_results = os.environ.get("ANALYZER_RESULTS_JSON")

candidates = []
if _env_results:
    candidates.append(Path(_env_results))

candidates += [
    Path("../../results.json"),
    Path("../results.json"),
    Path("results.json"),
    Path("../../results(1).json"),
    Path("../results(1).json"),
    Path("results(1).json"),
]

RESULTS_FILE = next((p for p in candidates if p.exists()), None)

if RESULTS_FILE is None:
    raise FileNotFoundError(
        "No se encontró results.json. Define ANALYZER_RESULTS_JSON o ajusta la lista 'candidates'."
    )

print("Archivo utilizado:", RESULTS_FILE.resolve())

with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

## 3. Resumen general de la ejecución

In [ ]:
summary = data["summary"]

for key, value in summary.items():
    print(f"{key}: {value}")


Este resumen corresponde a la ejecución completa del Miner. El campo `total_findings` representa el total integrado de hallazgos de todas las herramientas incluidas en el dataset.

## 4. Estado de los repositorios

In [ ]:
repo_status = pd.DataFrame([
    {
        "repository": repo.get("name"),
        "status": repo.get("status"),
        "languages": ", ".join(repo.get("languages", []) or []),
        "analyzed_languages": ", ".join(repo.get("analyzed_languages", []) or []),
        "codeql_findings": len(repo.get("findings", []) or []),
        "error": repo.get("error"),
        "sbom_status": (repo.get("sbom") or {}).get("status"),
        "sbom_components": (repo.get("sbom") or {}).get("component_count"),
        "grype_status": (repo.get("grype") or {}).get("status"),
        "grype_vulnerabilities": (repo.get("grype") or {}).get("vulnerability_count"),
    }
    for repo in data.get("repositories", [])
])

repo_status


In [ ]:
repo_status["status"].value_counts()


In [ ]:
repo_status[
    ~repo_status["status"].isin(["analyzed", "unsupported"])
][["repository", "status", "error"]]


Los repositorios fallidos deben interpretarse como ejecuciones incompletas, no como repositorios sin hallazgos. Esta diferencia es importante para evitar sesgos en la comparación.

## 5. Dataset integrado de hallazgos

In [ ]:
df_all = pd.DataFrame(data.get("findings", []))

print("Filas del dataset integrado:", len(df_all))
print("Columnas:", list(df_all.columns))
print("Dimensiones:", df_all.shape)

df_all.head()


### Validación del total integrado

In [ ]:
declared_total = data["summary"].get("total_findings")

print("Hallazgos en data['findings']:", len(df_all))
print("Hallazgos declarados por Miner:", declared_total)

assert len(df_all) == declared_total

print("\nDataset integrado cargado correctamente.")


## 6. Distribución por herramienta

In [ ]:
tool_counts = df_all["tool"].value_counts()

tool_percentages = (
    df_all["tool"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

tool_summary = pd.DataFrame({
    "cantidad": tool_counts,
    "porcentaje": tool_percentages
})

tool_summary


In [ ]:
tool_counts.plot(kind="bar", figsize=(7, 5))

plt.title("Hallazgos por herramienta")
plt.xlabel("Herramienta")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


Esta separación permite distinguir correctamente entre problemas detectados en el código fuente y vulnerabilidades conocidas en dependencias.

## 7. Separación de CodeQL y Grype

In [ ]:
df_codeql = df_all[
    df_all["tool"].str.lower() == "codeql"
].copy()

df_grype = df_all[
    df_all["tool"].str.lower() == "grype"
].copy()

print("Hallazgos CodeQL:", len(df_codeql))
print("Vulnerabilidades Grype:", len(df_grype))
print("Total:", len(df_codeql) + len(df_grype))


## 8. Análisis CodeQL — severidad

In [ ]:
codeql_severity = df_codeql["severity"].value_counts()

codeql_severity_pct = (
    df_codeql["severity"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

pd.DataFrame({
    "cantidad": codeql_severity,
    "porcentaje": codeql_severity_pct
})


In [ ]:
codeql_severity.plot(kind="bar", figsize=(7, 5))

plt.title("Severidad de hallazgos CodeQL")
plt.xlabel("Severidad")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Análisis CodeQL — tipos de hallazgo

In [ ]:
codeql_rules = df_codeql["vulnerability_type"].value_counts()

codeql_rules.head(20)


In [ ]:
codeql_rules.head(15).sort_values().plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("15 reglas CodeQL más frecuentes")
plt.xlabel("Cantidad")
plt.ylabel("Regla")
plt.tight_layout()
plt.show()


## 10. CodeQL — distribución por repositorio

In [ ]:
codeql_by_repo = df_codeql["repository"].value_counts()

codeql_by_repo.head(15)


In [ ]:
codeql_by_repo.head(10).sort_values().plot(
    kind="barh",
    figsize=(10, 6)
)

plt.title("Top 10 repositorios con más hallazgos CodeQL")
plt.xlabel("Cantidad")
plt.ylabel("Repositorio")
plt.tight_layout()
plt.show()


## 11. CodeQL — archivos de prueba vs otros archivos

In [ ]:
df_codeql["is_test"] = df_codeql["location"].str.contains(
    r"(?:^|/)(?:__tests__|__testfixtures__|test|tests|spec|specs|testing)(?:/|$)|(?:\.test\.|\.spec\.)",
    case=False,
    regex=True,
    na=False
)

test_counts = df_codeql["is_test"].value_counts()
test_pct = (
    df_codeql["is_test"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

pd.DataFrame({
    "cantidad": test_counts,
    "porcentaje": test_pct
}).rename(index={True: "Pruebas", False: "Otros archivos"})


Separar archivos de prueba evita interpretar suites de testing extensas como si fueran equivalentes al código utilizado en producción.

## 12. Grype — distribución por severidad

In [ ]:
grype_severity = df_grype["severity"].value_counts()

grype_severity_pct = (
    df_grype["severity"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

grype_severity_summary = pd.DataFrame({
    "cantidad": grype_severity,
    "porcentaje": grype_severity_pct
})

grype_severity_summary


In [ ]:
grype_severity.plot(kind="bar", figsize=(8, 5))

plt.title("Severidad de vulnerabilidades detectadas por Grype")
plt.xlabel("Severidad")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 13. Grype — identificadores de vulnerabilidad más frecuentes

In [ ]:
grype_vuln_types = df_grype["vulnerability_type"].value_counts()

grype_vuln_types.head(20)


In [ ]:
grype_vuln_types.head(15).sort_values().plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("15 vulnerabilidades Grype más frecuentes")
plt.xlabel("Cantidad de detecciones")
plt.ylabel("Vulnerabilidad")
plt.tight_layout()
plt.show()


Una detección no equivale necesariamente a una vulnerabilidad única. El mismo identificador puede aparecer en múltiples repositorios, paquetes o versiones.

## 14. Grype — paquetes más afectados

In [ ]:
grype_packages = df_grype["package"].value_counts()

grype_packages.head(20)


In [ ]:
grype_packages.head(15).sort_values().plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("15 paquetes con más detecciones Grype")
plt.xlabel("Cantidad de detecciones")
plt.ylabel("Paquete")
plt.tight_layout()
plt.show()


## 15. Grype — repositorios con más vulnerabilidades

In [ ]:
grype_by_repo = df_grype["repository"].value_counts()

grype_by_repo.head(15)


In [ ]:
grype_by_repo.head(10).sort_values().plot(
    kind="barh",
    figsize=(10, 6)
)

plt.title("Top 10 repositorios con más detecciones Grype")
plt.xlabel("Cantidad de detecciones")
plt.ylabel("Repositorio")
plt.tight_layout()
plt.show()


## 16. Grype — vulnerabilidades únicas

In [ ]:
unique_vulnerabilities = df_grype["vulnerability_type"].nunique()
unique_packages = df_grype["package"].nunique()

print("Vulnerabilidades únicas:", unique_vulnerabilities)
print("Paquetes afectados únicos:", unique_packages)


## 17. Grype — versiones instaladas y corregidas

In [ ]:
fixed_available = df_grype["fixed_version"].notna() & df_grype["fixed_version"].astype(str).str.strip().ne("")

fixed_summary = pd.Series({
    "con_version_corregida": int(fixed_available.sum()),
    "sin_version_corregida": int((~fixed_available).sum())
})

fixed_summary


In [ ]:
fixed_percentage = fixed_available.mean() * 100 if len(df_grype) else 0

print(f"Detecciones con versión corregida disponible: {fixed_percentage:.2f}%")


## 18. Grype — versiones con mayor exposición

In [ ]:
package_version_exposure = (
    df_grype.groupby(["package", "version"], dropna=False)
    .size()
    .rename("detections")
    .reset_index()
    .sort_values("detections", ascending=False)
)

package_version_exposure.head(25)


## 19. Grype — severidad por repositorio

In [ ]:
grype_repo_severity = pd.crosstab(
    df_grype["repository"],
    df_grype["severity"]
)

grype_repo_severity["total"] = grype_repo_severity.sum(axis=1)

grype_repo_severity.sort_values("total", ascending=False).head(15)


## 20. Índice simple de priorización Grype

In [ ]:
severity_weights = {
    "Critical": 4,
    "High": 3,
    "Medium": 2,
    "Low": 1,
    "Negligible": 0.5,
    "Unknown": 0
}

df_grype["severity_weight"] = (
    df_grype["severity"]
    .map(severity_weights)
    .fillna(0)
)

repo_grype_priority = (
    df_grype.groupby("repository")
    .agg(
        detections=("vulnerability_type", "size"),
        unique_vulnerabilities=("vulnerability_type", "nunique"),
        affected_packages=("package", "nunique"),
        priority_score=("severity_weight", "sum")
    )
    .sort_values("priority_score", ascending=False)
)

repo_grype_priority.head(15)


El `priority_score` es una métrica interna de priorización y no una métrica estándar. Se usa solo para comparar repositorios ponderando severidades.

## 21. Relación entre CodeQL y Grype por repositorio

In [ ]:
repo_integrated = pd.DataFrame(index=sorted(set(df_all["repository"].dropna())))

repo_integrated["codeql_findings"] = codeql_by_repo
repo_integrated["grype_detections"] = grype_by_repo

repo_integrated = repo_integrated.fillna(0).astype(int)

repo_integrated["total_security_evidence"] = (
    repo_integrated["codeql_findings"]
    + repo_integrated["grype_detections"]
)

repo_integrated.sort_values(
    "total_security_evidence",
    ascending=False
).head(15)


Esta comparación muestra que un repositorio puede destacar por problemas de código, por vulnerabilidades de dependencias, o por ambas fuentes. No deben sumarse conceptualmente como si fueran el mismo tipo de hallazgo, pero sí pueden compararse como evidencia complementaria.

## 22. Concentración de Grype

In [ ]:
grype_concentration = pd.DataFrame({
    "detections": grype_by_repo
})

grype_concentration["percentage"] = (
    grype_concentration["detections"]
    / grype_concentration["detections"].sum()
    * 100
).round(2)

grype_concentration["cumulative_percentage"] = (
    grype_concentration["percentage"]
    .cumsum()
    .round(2)
)

grype_concentration.head(15)


In [ ]:
top3_grype_share = (
    grype_by_repo.head(3).sum() / len(df_grype) * 100
    if len(df_grype) else 0
)

top5_grype_share = (
    grype_by_repo.head(5).sum() / len(df_grype) * 100
    if len(df_grype) else 0
)

print(f"Top 3 repositorios: {top3_grype_share:.2f}%")
print(f"Top 5 repositorios: {top5_grype_share:.2f}%")


## 23. Conclusiones automáticas

In [ ]:
codeql_count = len(df_codeql)
grype_count = len(df_grype)
total_count = len(df_all)

codeql_pct = codeql_count / total_count * 100 if total_count else 0
grype_pct = grype_count / total_count * 100 if total_count else 0

critical_count = int((df_grype["severity"] == "Critical").sum())
high_count = int((df_grype["severity"] == "High").sum())

top_grype_repo = grype_by_repo.index[0] if not grype_by_repo.empty else "N/A"
top_grype_repo_count = int(grype_by_repo.iloc[0]) if not grype_by_repo.empty else 0

top_package = grype_packages.index[0] if not grype_packages.empty else "N/A"
top_package_count = int(grype_packages.iloc[0]) if not grype_packages.empty else 0

top_codeql_repo = codeql_by_repo.index[0] if not codeql_by_repo.empty else "N/A"
top_codeql_repo_count = int(codeql_by_repo.iloc[0]) if not codeql_by_repo.empty else 0

text = (
    "### Principales observaciones\n\n"
    f"- El dataset integrado contiene **{total_count} hallazgos**.\n"
    f"- **{codeql_count} ({codeql_pct:.2f}%)** corresponden a CodeQL.\n"
    f"- **{grype_count} ({grype_pct:.2f}%)** corresponden a Grype.\n"
    f"- Grype identificó **{critical_count} detecciones Critical** y **{high_count} High**.\n"
    f"- Se observaron **{unique_vulnerabilities} vulnerabilidades Grype únicas** en **{unique_packages} paquetes afectados**.\n"
    f"- **{fixed_percentage:.2f}%** de las detecciones Grype presentan una versión corregida disponible.\n"
    f"- El repositorio con más detecciones Grype es **`{top_grype_repo}`**, con **{top_grype_repo_count}**.\n"
    f"- El paquete con más detecciones Grype es **`{top_package}`**, con **{top_package_count}**.\n"
    f"- El repositorio con más hallazgos CodeQL es **`{top_codeql_repo}`**, con **{top_codeql_repo_count}**.\n"
    f"- Los tres repositorios con más detecciones Grype concentran **{top3_grype_share:.2f}%** del total de Grype.\n\n"
    
)

display(Markdown(text))


## 24. Exportación de resultados

In [ ]:
import os

OUTPUT_DIR = Path(os.environ.get("ANALYZER_OUTPUT_DIR", "../output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_all.to_csv(
    OUTPUT_DIR / "integrated_findings.csv",
    index=False
)

df_codeql.to_csv(
    OUTPUT_DIR / "codeql_findings.csv",
    index=False
)

df_grype.to_csv(
    OUTPUT_DIR / "grype_findings.csv",
    index=False
)

repo_integrated.reset_index().rename(
    columns={"index": "repository"}
).to_csv(
    OUTPUT_DIR / "repository_integrated_summary.csv",
    index=False
)

repo_grype_priority.reset_index().to_csv(
    OUTPUT_DIR / "grype_repository_priority.csv",
    index=False
)

grype_concentration.reset_index().to_csv(
    OUTPUT_DIR / "grype_concentration.csv",
    index=False
)

grype_severity_summary.reset_index().rename(
    columns={"index": "severity"}
).to_json(
    OUTPUT_DIR / "grype_severity_summary.json",
    orient="records",
    indent=2
)

repo_integrated.reset_index().rename(
    columns={"index": "repository"}
).to_json(
    OUTPUT_DIR / "repository_integrated_summary.json",
    orient="records",
    indent=2
)

print("Resultados exportados en:", OUTPUT_DIR.resolve())

## 25. Conclusión metodológica

El `results.json` integrado permite analizar en un mismo dataset dos dimensiones distintas de seguridad:

- **CodeQL** entrega evidencia sobre problemas identificados en el código fuente.
- **Grype** entrega evidencia sobre vulnerabilidades conocidas en dependencias y componentes.

Ambas fuentes deben interpretarse por separado antes de compararse, porque miden fenómenos distintos.

El Analyzer debe buscar no solo conteos, sino también:

- diferencias entre repositorios;
- concentración de hallazgos;
- severidades;
- tipos de vulnerabilidad;
- paquetes recurrentemente afectados;
- disponibilidad de versiones corregidas;
- relaciones entre evidencia de código y evidencia de dependencias.

